# Criptografia na Prática · Caderno do aluno
**WorkInTech · Axion** · oficina de Bruno Fernandes Ortega

Antes de tudo:

1. **Arquivo → Salvar uma cópia no Drive.** Sem isso, o que você escrever some quando fechar a aba, e amanhã vamos reaproveitar o código de hoje.
2. Rode as células **de cima para baixo**, com **Shift + Enter**.
3. Onde o slide mostra `???`, aqui aparece `None` com um comentário `# TODO`. Troque o `None` pela sua resposta e rode a célula de novo.
4. Depois de cada exercício há uma célula **Conferir**: ela diz ✓ quando está certo ou mostra o que saiu.

In [ ]:
# Prepara o caderno: rode esta célula primeiro.
from collections import Counter

def curto(valor):
    texto = repr(valor)
    return texto if len(texto) <= 60 else texto[:60] + "…"

def confere(descricao, funcao, esperado):
    """Roda funcao() e compara com o esperado. Mostra ✓ ou explica o que saiu."""
    try:
        obtido = funcao()
    except Exception as erro:
        print(f"✗ {descricao}: deu erro ({type(erro).__name__}: {erro}). Complete o TODO e rode a célula da função de novo.")
        return False
    if obtido == esperado:
        print(f"✓ {descricao}")
        return True
    print(f"✗ {descricao}: esperava {curto(esperado)}, saiu {curto(obtido)}")
    return False

print("Pronto. Pode seguir.")

## Dia 1 · Cifra de César
Cada letra vira um número (A = 0 … Z = 25), soma k e volta para letra: **C = (P + k) mod 26**. Slide 17.

In [ ]:
def cesar(texto, k):
    resultado = ""
    for c in texto.upper():
        if c.isalpha():
            n = ord(c) - ord("A")       # A = 0 ... Z = 25
            n = None                    # TODO: some k e use mod 26
            resultado += chr(n + ord("A"))
        else:
            resultado += c
    return resultado

In [ ]:
# Conferir
confere('cesar("OLA MUNDO", 3)', lambda: cesar("OLA MUNDO", 3), "ROD PXQGR")
confere('cesar("ROD PXQGR", -3) (decifrar é cifrar com -k)', lambda: cesar("ROD PXQGR", -3), "OLA MUNDO")
confere('cesar("Z", 3) dá a volta no alfabeto', lambda: cesar("Z", 3), "C")

### Quebrando o César: força bruta (slide 18 · nível 1 do CTF)
Só existem 25 chaves úteis. Teste todas e leia qual linha é português. No CTF, troque `msg` pela mensagem do seu puzzle.

In [ ]:
msg = "ROD PXQGR"   # cole aqui a mensagem cifrada
for k in range(1, 26):
    print(k, None)              # TODO: decifre msg com k

### Ensine o Python a reconhecer português (slide 19)
Em português, A, E e O são as letras mais comuns. A chave certa é a que deixa o texto decifrado com mais A, E e O.

In [ ]:
def pontuacao(texto):
    return None                    # TODO: conte os A, E e O do texto

def quebrar_cesar(texto):
    melhor_k = 0
    melhor_pontos = 0
    for k in range(26):
        pontos = pontuacao(cesar(texto, -k))
        if pontos > melhor_pontos:
            melhor_k = k
            melhor_pontos = pontos
    return melhor_k

In [ ]:
# Conferir
confere('pontuacao("ABACATE")', lambda: pontuacao("ABACATE"), 4)
confere('quebrar_cesar acha k = 7', lambda: quebrar_cesar(cesar("O CODIGO SECRETO ESTA AQUI", 7)), 7)

## Dia 1 · Cifra de Vigenère (slide 21)
Um César cujo k muda a cada letra, seguindo uma palavra-chave que se repete. Reaproveite o `cesar()`.

In [ ]:
def vigenere(texto, chave):
    resultado = ""
    i = 0                                   # posição na chave
    for c in texto.upper():
        if c.isalpha():
            k = ord(chave[i % len(chave)]) - ord("A")
            resultado += None           # TODO: reaproveite o cesar()
            i += 1
        else:
            resultado += c
    return resultado

In [ ]:
# Conferir
confere('vigenere("ATTACKATDAWN", "LEMON")', lambda: vigenere("ATTACKATDAWN", "LEMON"), "LXFOPVEFRNHR")
confere('espaços não gastam letras da chave', lambda: vigenere("ATTACK AT DAWN", "LEMON"), "LXFOPV EF RNHR")

## Dia 2 · Quebrando o Vigenère
Abra **a sua cópia salva ontem**. O Python esqueceu tudo durante a noite, então escolha um caminho:

- **Terminou o Dia 1?** Rode de novo, em ordem, a primeira célula e as células das suas funções (`cesar`, `pontuacao` e `quebrar_cesar`, `vigenere`).
- **Não terminou ou deu erro?** Rode só a célula **Plano B** logo abaixo: ela carrega tudo pronto.

In [ ]:
# @title Plano B: carregar as funções do Dia 1 prontas { display-mode: "form" }
from collections import Counter

def curto(valor):
    texto = repr(valor)
    return texto if len(texto) <= 60 else texto[:60] + "…"

def confere(descricao, funcao, esperado):
    """Roda funcao() e compara com o esperado. Mostra ✓ ou explica o que saiu."""
    try:
        obtido = funcao()
    except Exception as erro:
        print(f"✗ {descricao}: deu erro ({type(erro).__name__}: {erro}). Complete o TODO e rode a célula da função de novo.")
        return False
    if obtido == esperado:
        print(f"✓ {descricao}")
        return True
    print(f"✗ {descricao}: esperava {curto(esperado)}, saiu {curto(obtido)}")
    return False

def cesar(texto, k):
    resultado = ""
    for c in texto.upper():
        if c.isalpha():
            n = (ord(c) - ord("A") + k) % 26
            resultado += chr(n + ord("A"))
        else:
            resultado += c
    return resultado

def pontuacao(texto):
    return texto.count("A") + texto.count("E") + texto.count("O")

def quebrar_cesar(texto):
    melhor_k, melhor_pontos = 0, 0
    for k in range(26):
        pontos = pontuacao(cesar(texto, -k))
        if pontos > melhor_pontos:
            melhor_k, melhor_pontos = k, pontos
    return melhor_k

def vigenere(texto, chave):
    resultado, i = "", 0
    for c in texto.upper():
        if c.isalpha():
            resultado += cesar(c, ord(chave[i % len(chave)]) - ord("A"))
            i += 1
        else:
            resultado += c
    return resultado

print("Funções do Dia 1 carregadas.")

### Passo 1: o tamanho da chave (slide 25)
O **índice de coincidência** (IC) é a chance de duas letras sorteadas do texto serem iguais: perto de 0,07 em português, perto de 0,04 em texto embaralhado. Para cada tamanho m, separe o texto em m colunas e tire a média do IC. O m certo e os múltiplos dele dão o IC mais alto. Como ler a tabela:

1. Ache o **maior IC** (fica perto de 0,08).
2. A resposta é o **menor m que chega perto dele** (diferença menor que 0,01).
3. Valores intermediários, como 0,06, **não contam**: aparecem quando m divide o tamanho real (chave de 6 letras faz o m = 3 subir um pouco).

O texto de exemplo abaixo foi cifrado com uma chave de 3 letras. No CTF, cole o seu texto no lugar.

In [ ]:
def ic(s):
    n = len(s)
    return sum(v * (v - 1) for v in Counter(s).values()) / (n * (n - 1))

texto = "FCDWQ FDCOW NPKGP AGLUW QJOOW JTYSY WFPYO YZCFS TLEOO WWYVS NATCS JPDSW SFPKW DLWFH CCIIL KSEJS KWBEG GLFCD SHPII PTOMT ORWSV SGTKY THSCU SMWFL EEFWO NZOGW GPJSA WHPWE FWQLV ONGZF FOOGH PPHZW OAWBL KIXSQ TXFLV SNWGL JVZBS GGQPN OTJSQ SNPJS DKSLL OBMSN GAAGI NSGWA BSSGO WDJLV ZFSXM WESSD LOEAG EAQL"
texto = texto.replace(" ", "")      # tire os espaços antes de separar as colunas!

for m in range(2, 9):
    colunas = None                  # TODO: m colunas (dica: texto[i::m])
    media = sum(ic(c) for c in colunas) / m
    print(m, round(media, 4))

In [ ]:
# Conferir (depois de rodar a célula acima)
confere('ic("BANANA")', lambda: round(ic("BANANA"), 4), 0.2667)
confere('as 8 colunas do último passo do laço (m = 8)', lambda: colunas, [texto[i::8] for i in range(8)])

### Passo 2: a chave, coluna por coluna (slide 26)
Cada coluna é um César, e César você já quebra com `quebrar_cesar`. Se a chave sair repetida (ex.: SOLSOL), a verdadeira é a menor.

In [ ]:
m = 3                                   # o tamanho que você descobriu
chave = ""
for i in range(m):
    coluna = texto[i::m]
    k = quebrar_cesar(coluna)           # o ataque ao César de ontem!
    chave += None                   # TODO: transforme k na letra da chave
print(chave)

In [ ]:
# Conferir (com o texto de exemplo)
confere('chave do texto de exemplo', lambda: chave, "SOL")

## Dia 2 · Euclides e RSA
### Dois truques (slide 27)

In [ ]:
def mdc(a, b):
    while b:
        a, b = b, a % b           # as duas atribuições acontecem ao mesmo tempo
    return a

print(mdc(48, 18))                # 6
print(pow(17, -1, 3120))          # 2753: o inverso de 17 módulo 3120
print(17 * 2753 % 3120)           # 1

### RSA em poucas linhas (slide 30)
Troque p e q por outros primos pequenos e veja o que muda.

In [ ]:
p, q = 61, 53
n = p * q                         # 3233
phi = (p - 1) * (q - 1)           # 3120
e = 17
d = pow(e, -1, phi)               # 2753: a chave privada
c = pow(65, e, n)                 # cifrar o 65: dá 2790
m = pow(c, d, n)                  # decifrar: volta o 65
print(n, phi, d, c, m)

### Quebrando o RSA: fatorar n (slide 31 · nível 3 do CTF)
Quem fatora n = p · q recalcula φ(n) e acha d. No CTF, troque n, e e c pelos do seu puzzle.

In [ ]:
def fatorar(n):
    f = 2
    while None:                     # TODO: enquanto f NÃO divide n
        f += 1
    return f, n // f

n, e, c = 3233, 17, 2790          # cole aqui os valores do seu puzzle

p, q = fatorar(n)
phi = None                          # TODO: φ(n)
d = pow(e, -1, phi)
m = pow(c, d, n)
print(m)

In [ ]:
# Conferir
confere('fatorar(3233)', lambda: fatorar(3233), (53, 61))
confere('p, q, φ(n) e a mensagem do exemplo', lambda: (p, q, phi, m), (53, 61, 3120, 65))

## CTF · como usar este caderno
O endereço e o QR code estão no slide. Cada apelido recebe puzzles próprios: copiar a resposta do colega não funciona.

| Nível | Use a célula | Formato da flag |
| --- | --- | --- |
| 1 · César | força bruta do Dia 1 (troque `msg`) | `FLAG{TERMO-ANIMAL}` |
| 2 · Vigenère | passos 1 e 2 do Dia 2 (troque `texto` e depois `m`) | `FLAG{CHAVE}` |
| 3 · RSA | quebrando o RSA (troque `n`, `e` e `c`) | `FLAG{m}` |

Errou? A espera cresce a cada erro (3 s, depois 30 s, depois 2 min). Confira o formato antes de enviar.